# Assignment No. 3: Exploratory Data Analysis (EDA)
## Introduction to Data Science

**Student Name:** Ali Raza  
**Roll No.:** 014  
**Section:** BSCS 3A  
**Dataset:** Student Performance (Mathematics)  
**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/320/student+performance)

---

## 1. Dataset Introduction

### Objective
The purpose of this analysis is to explore student performance data and identify patterns between final grades and factors such as study time, absences, previous grades, age, and family support.

### Research question
**Which student-related factors show relationships with the final mathematics grade (`G3`)?**

The target variable is `G3`, the final mathematics grade on a scale from 0 to 20. This is an exploratory analysis; relationships do not necessarily prove cause and effect.

### Load libraries and dataset

The code below downloads the official UCI dataset archive and loads the mathematics data. If the download fails, download `student.zip` from the UCI page linked above and upload it to the notebook.

In [ ]:
import io
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

try:
    with urllib.request.urlopen(url, timeout=30) as response:
        zip_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        csv_name = next(name for name in z.namelist() if name.endswith("student-mat.csv"))
        df = pd.read_csv(z.open(csv_name), sep=";")
except Exception as e:
    print("Automatic download failed:", e)
    from google.colab import files
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith(".zip")), None)
    csv_name = next((name for name in uploaded if name.endswith(".csv")), None)
    if zip_name:
        with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
            member = next(name for name in z.namelist() if name.endswith("student-mat.csv"))
            df = pd.read_csv(z.open(member), sep=";")
    elif csv_name:
        df = pd.read_csv(io.BytesIO(uploaded[csv_name]), sep=";")
    else:
        raise ValueError("Please upload the UCI student.zip file or student-mat.csv.")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

### Dataset overview

The original UCI repository provides student records for mathematics and Portuguese language courses. This notebook analyzes the **mathematics file** (`student-mat.csv`) only. It contains demographic, family, school, study, attendance, and grade information.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nStatistical summary:")
display(df.describe(include="all").T)

## 2. Data Quality Check

Check missing values, duplicate rows, and basic data quality before interpreting the graphs.

In [ ]:
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

print("Duplicate rows:", df.duplicated().sum())
print("Final grade range:", df["G3"].min(), "to", df["G3"].max())

**Interpretation:** The missing-value table identifies columns that may require cleaning. The duplicate count indicates whether identical rows are present. The grade range confirms the observed limits of the target variable. The original dataset is documented by UCI as having no missing values.

## 3. Univariate EDA

Univariate analysis examines one variable at a time to understand its distribution, typical values, spread, and unusual observations.

### 3.1 Distribution of final grade (`G3`)

In [ ]:
sns.histplot(data=df, x="G3", bins=20, kde=True)
plt.title("Distribution of Final Mathematics Grades (G3)")
plt.xlabel("Final grade (0–20)")
plt.ylabel("Number of students")
plt.show()

print("Mean final grade:", round(df["G3"].mean(), 2))
print("Median final grade:", round(df["G3"].median(), 2))

**Interpretation:** The histogram shows how frequently different final grades occur. The mean and median summarize the center of the distribution. Use the plotted distribution to comment on whether grades cluster around particular values or show a long tail.

### 3.2 Study time

In [ ]:
sns.countplot(data=df, x="studytime", order=sorted(df["studytime"].unique()))
plt.title("Weekly Study-Time Categories")
plt.xlabel("Study time category (1 = less than 2 hours; 4 = more than 10 hours)")
plt.ylabel("Number of students")
plt.show()

display(df["studytime"].value_counts().sort_index().rename("Students").to_frame())

**Interpretation:** This bar chart compares the number of students in each weekly study-time category. The categories are ordinal, so higher numbers indicate more reported study time.

### 3.3 Absences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="absences", bins=25, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Absences")
sns.boxplot(data=df, x="absences", ax=axes[1])
axes[1].set_title("Boxplot of Absences")
plt.tight_layout()
plt.show()

display(df["absences"].describe().to_frame("Absences summary"))

**Interpretation:** The histogram shows how absences are distributed, while the boxplot highlights the median, spread, and possible outliers. A small number of students with many absences may make the distribution right-skewed.

### 3.4 Age and previous grades

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=df, x="age", discrete=True, ax=axes[0])
axes[0].set_title("Student Age")
sns.histplot(data=df, x="G1", bins=15, kde=True, ax=axes[1])
axes[1].set_title("First-Period Grade (G1)")
sns.histplot(data=df, x="G2", bins=15, kde=True, ax=axes[2])
axes[2].set_title("Second-Period Grade (G2)")
plt.tight_layout()
plt.show()

**Interpretation:** These plots compare the distributions of age and the two earlier grades. `G1` and `G2` are useful for understanding how earlier academic performance relates to the final grade.

## 4. Target Analysis

The target variable is `G3`, the final mathematics grade. For this descriptive analysis, grades are grouped into three bands: **Low (0–9), Medium (10–14), and High (15–20)**. These bands are used only to summarize the target distribution, not as official pass/fail definitions.

In [ ]:
df["grade_band"] = pd.cut(
    df["G3"],
    bins=[-1, 9, 14, 20],
    labels=["Low (0–9)", "Medium (10–14)", "High (15–20)"]
)
grade_counts = df["grade_band"].value_counts().reindex(["Low (0–9)", "Medium (10–14)", "High (15–20)"])
grade_percent = (grade_counts / grade_counts.sum() * 100).round(1)

ax = grade_counts.plot(kind="bar", rot=0)
plt.title("Final Grade Bands")
plt.xlabel("Grade band")
plt.ylabel("Number of students")
plt.show()

display(pd.DataFrame({"Students": grade_counts, "Percentage": grade_percent}))

**Interpretation:** Compare the counts and percentages across the three bands. If one band contains substantially more students than the others, the grouped target is imbalanced. The original numeric grade itself is a regression target; class balance applies only after grouping it into bands.

## 5. Bivariate EDA

Bivariate analysis examines relationships between two variables. These are associations in the observed data and do not prove that one factor causes another.

### 5.1 Earlier grades vs final grade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=df, x="G1", y="G3", scatter_kws={"alpha": 0.55}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_title("First-Period Grade vs Final Grade")
sns.regplot(data=df, x="G2", y="G3", scatter_kws={"alpha": 0.55}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_title("Second-Period Grade vs Final Grade")
plt.tight_layout()
plt.show()

print("Correlation G1 vs G3:", round(df["G1"].corr(df["G3"]), 3))
print("Correlation G2 vs G3:", round(df["G2"].corr(df["G3"]), 3))

**Interpretation:** An upward pattern means students with higher earlier grades tend also to have higher final grades. The correlation values quantify the strength of the linear association. Note that earlier grades may make final-grade prediction easier and should be considered carefully when designing a real prediction system.

### 5.2 Study time vs final grade

In [ ]:
sns.boxplot(data=df, x="studytime", y="G3", order=sorted(df["studytime"].unique()))
plt.title("Final Grade by Weekly Study-Time Category")
plt.xlabel("Study time category")
plt.ylabel("Final grade (G3)")
plt.show()

display(df.groupby("studytime", observed=False)["G3"].agg(["count", "mean", "median"]).round(2))

**Interpretation:** Compare the median lines and distributions across study-time categories. If grade levels tend to rise with study time, this suggests a positive association in this dataset; differences between categories may overlap and do not establish causation.

### 5.3 Absences vs final grade

In [ ]:
sns.scatterplot(data=df, x="absences", y="G3", alpha=0.7)
plt.title("Absences vs Final Grade")
plt.xlabel("Number of absences")
plt.ylabel("Final grade (G3)")
plt.show()

print("Correlation absences vs G3:", round(df["absences"].corr(df["G3"]), 3))

**Interpretation:** Look for a general upward or downward pattern and for students with unusually high absence counts. The correlation summarizes a linear relationship but may not capture every pattern.

### 5.4 Final grade by school and sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, x="school", y="G3", ax=axes[0])
axes[0].set_title("Final Grade by School")
sns.boxplot(data=df, x="sex", y="G3", ax=axes[1])
axes[1].set_title("Final Grade by Sex")
plt.tight_layout()
plt.show()

display(df.groupby("school")["G3"].agg(["count", "mean", "median"]).round(2))
display(df.groupby("sex")["G3"].agg(["count", "mean", "median"]).round(2))

**Interpretation:** The boxplots compare grade distributions between categories. Compare medians and spread rather than relying only on averages. These group differences describe this dataset and should not be generalized automatically to all students.

## 6. Correlation Analysis

The correlation heatmap displays linear relationships between numerical variables. Values near +1 indicate strong positive linear association, values near −1 indicate strong negative linear association, and values near 0 indicate weak linear association.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

plt.figure(figsize=(15, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", square=False)
plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.show()

target_corr = corr["G3"].drop("G3").sort_values(key=lambda s: s.abs(), ascending=False)
print("Numerical variables most strongly correlated with G3:")
display(target_corr.head(10).to_frame("Correlation with G3"))

**Interpretation:** Use the heatmap and ranked table to identify the strongest positive and negative associations with `G3`. In this dataset, `G1` and `G2` are expected to be strongly associated with `G3`, because they are earlier grades from the same school year. Correlation does not imply causation.

## 7. Multivariate EDA

This section explores several variables together to identify combined patterns.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="G1", y="G2", hue="studytime",
    size="absences", sizes=(25, 180), alpha=0.75
)
plt.title("G1 vs G2: Study Time and Absences Together")
plt.xlabel("First-period grade (G1)")
plt.ylabel("Second-period grade (G2)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation:** Each point represents a student. The color shows study-time category and point size represents absences. The plot helps inspect whether earlier grades move together and whether study time or absences show visible patterns within that relationship.

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x="studytime", y="G3", hue="schoolsup")
plt.title("Final Grade by Study Time and Extra School Support")
plt.xlabel("Study-time category")
plt.ylabel("Final grade (G3)")
plt.tight_layout()
plt.show()

**Interpretation:** This chart compares final-grade distributions by study time while also separating students who did or did not receive extra educational support. Group sizes may differ, so interpret apparent differences cautiously.

## 8. Patterns, Trends, and Anomalies

In [ ]:
# Use the IQR rule to flag possible outliers in selected numerical variables.
def iqr_outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = series[(series < lower) | (series > upper)]
    return {
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower bound": lower, "Upper bound": upper,
        "Potential outliers": len(flagged),
        "Example values": flagged.head(10).tolist()
    }

outlier_results = pd.DataFrame({
    "absences": iqr_outlier_summary(df["absences"]),
    "G3": iqr_outlier_summary(df["G3"])
}).T
display(outlier_results)

**Interpretation:** The IQR rule flags observations outside 1.5 times the interquartile range from the quartiles. These are **potential outliers**, not necessarily errors. For example, a high absence count may be a genuine student record and should not be removed without a valid reason.

## 9. Final Findings

Run the summary cell below to obtain data-specific statistics, then use them to complete the findings. Avoid claiming a pattern unless it is supported by the notebook's output.

In [ ]:
print(f"Number of students: {len(df)}")
print(f"Number of columns: {df.shape[1]}")
print(f"Mean final grade (G3): {df['G3'].mean():.2f}")
print(f"Median final grade (G3): {df['G3'].median():.2f}")
print(f"Mean absences: {df['absences'].mean():.2f}")
print(f"Correlation G1-G3: {df['G1'].corr(df['G3']):.3f}")
print(f"Correlation G2-G3: {df['G2'].corr(df['G3']):.3f}")
print(f"Correlation studytime-G3: {df['studytime'].corr(df['G3']):.3f}")
print(f"Correlation absences-G3: {df['absences'].corr(df['G3']):.3f}")
print("\nGrade-band percentages:")
display((df["grade_band"].value_counts(normalize=True) * 100).round(1).to_frame("Percent"))

### Key insights to report (verify against your outputs)

1. **Previous academic performance:** Compare the correlations of `G1` and `G2` with `G3` to identify how strongly earlier grades are associated with the final grade.
2. **Study time:** Compare the median and mean `G3` across study-time categories; do not assume the relationship is perfectly increasing.
3. **Attendance:** Use the scatter plot and correlation to describe the relationship between absences and final grade.
4. **Grade distribution:** Describe where final grades cluster and whether the distribution is skewed.
5. **Target balance:** Report the percentages in the low, medium, and high grade bands and identify whether any band dominates.
6. **Potential anomalies:** Report the number of IQR-flagged absence and grade observations. Explain that outliers need investigation rather than automatic deletion.
7. **Group comparisons:** Summarize any visible differences by school or sex, while noting that this dataset alone cannot explain the reasons for those differences.
8. **Multivariate patterns:** Describe any patterns visible when study time, absences, earlier grades, and school support are viewed together.

### How findings can help future modeling

The EDA helps choose candidate features, identify unusual values, understand target distribution, and decide which relationships deserve further testing. For a fair prediction task, avoid using information that would not be available at prediction time. In particular, `G1` and `G2` are powerful predictors of `G3` but may not be available for an early-year prediction. Any preprocessing, including scaling or imputation, should be fitted on the training data only.

## 10. Conclusion

This EDA examined the mathematics student dataset through univariate, bivariate, correlation, and multivariate analysis. The visualizations and summary statistics help describe grade distributions, relationships between academic and behavioral variables, and potential anomalies. The results are descriptive and should be validated on additional data before making decisions or building a predictive model.

## Dataset Reference

Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T.  
Dataset page: https://archive.ics.uci.edu/dataset/320/student+performance